In [7]:
import sys
import sklearn
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import os
import allel

df_2023 = pd.read_csv('g2f_2023_phenotypic_clean_data.csv', encoding='latin-1')
df_2022 = pd.read_csv('g2f_2022_phenotypic_clean_data.csv', encoding='latin-1')
df_2021 = pd.read_csv('g2f_2021_phenotypic_clean_data.csv', encoding='latin-1')
df_2020 = pd.read_csv('g2f_2020_phenotypic_clean_data.csv', encoding='latin-1')
df_2019 = pd.read_csv('g2f_2019_phenotypic_clean_data.csv', encoding='latin-1')
weather_2023 = pd.read_csv('G2F_WeatherData_2023.csv')
weather_2022 = pd.read_csv('G2F_WeatherData_2022.csv')
weather_2021 = pd.read_csv('G2F_WeatherData_2021.csv')
weather_2020 = pd.read_csv('G2F_WeatherData_2020.csv')
weather_2019 = pd.read_csv('G2F_WeatherData_2019.csv')
weather_2018 = pd.read_csv('G2F_WeatherData_2018.csv')
weather_2017 = pd.read_csv('G2F_WeatherData_2017.csv')
weather_2016 = pd.read_csv('G2F_WeatherData_2016.csv')
weather_2015 = pd.read_csv('G2F_WeatherData_2015.csv')
weather_2014 = pd.read_csv('G2F_WeatherData_2014.csv')

weather_dict = {}
weather_dict['2014'] = weather_2014
weather_dict['2015'] = weather_2015
weather_dict['2016'] = weather_2016
weather_dict['2017'] = weather_2017
weather_dict['2018'] = weather_2018
weather_dict['2019'] = weather_2019
weather_dict['2020'] = weather_2020
weather_dict['2021'] = weather_2021
weather_dict['2022'] = weather_2022
weather_dict['2023'] = weather_2023


/var/folders/yf/fmn9knz52b9fdxm6h9mn6qs80000gn/T/ipykernel_72100/682166993.py:10: DtypeWarning: Columns (0: Plot, 1: Pass) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2023 = pd.read_csv('g2f_2023_phenotypic_clean_data.csv', encoding='latin-1')
/var/folders/yf/fmn9knz52b9fdxm6h9mn6qs80000gn/T/ipykernel_72100/682166993.py:12: DtypeWarning: Columns (0: Plot) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2021 = pd.read_csv('g2f_2021_phenotypic_clean_data.csv', encoding='latin-1')
/var/folders/yf/fmn9knz52b9fdxm6h9mn6qs80000gn/T/ipykernel_72100/682166993.py:13: DtypeWarning: Columns (0: Pass, 1: Filler) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2020 = pd.read_csv('g2f_2020_phenotypic_clean_data.csv', encoding='latin-1')
/var/folders/yf/fmn9knz52b9fdxm6h9mn6qs80000gn/T/ipykernel_72100/682166993.py:14: DtypeWarning: Columns (0: Filler [enter 'filler' or blank], 1: Possible subs, 2: Confirm

In [8]:
required_phenotypes = [
    'Stand Count [# of plants]',
    'Plant Height [cm]',
    'Ear Height [cm]',
    'Anthesis [days]',
    'Silking [days]',
    'Grain Yield (bu/A)'
]

def clean_phenotypicData(df):
    df['experiment'] = df['Field-Location'].apply(lambda x: str(x)[2:] if pd.notna(x) and len(str(x)) >= 3 else None)
    df_h1 = df[df['experiment'] == 'H1'].copy()
    initial_count = len(df_h1)
    df_clean = df_h1.dropna(subset=required_phenotypes).copy()
    removed_count = initial_count - len(df_clean)
    return df_clean

cleaned_dfs = []
for year, df in [('2023', df_2023), ('2022', df_2022), 
                 ('2021', df_2021), ('2020', df_2020), ('2019', df_2019)]:
    cleaned = clean_phenotypicData(df)
    cleaned['year'] = year
    cleaned_dfs.append(cleaned)

full_phenotypic_df = pd.concat(cleaned_dfs, ignore_index=True)

In [9]:
dailyWeather_list = []

for year, df in weather_dict.items(): 
    df_copy = df.copy()
    
    df_copy['datetime'] = pd.to_datetime(
        df_copy['Year'].astype(str) + '-' + 
        df_copy['Month'].astype(str).str.zfill(2) + '-' + 
        df_copy['Day'].astype(str).str.zfill(2) + ' ' + 
        df_copy['Time'],
        format='%Y-%m-%d %H:%M',
        errors='coerce'
    )
    df_copy['date'] = df_copy['datetime'].dt.date
    df_copy['Field-Location'] = df_copy['Experiment'].str[:2] + 'H1'
    
    temp_col = [col for col in df_copy.columns if 'Temperature' in col][0]
    humid_col = [col for col in df_copy.columns if 'Relative_Humidity' in col][0]
    solar_col = [col for col in df_copy.columns if 'Solar' in col][0]
    rain_col = [col for col in df_copy.columns if 'Rainfall' in col][0]
    wind_col = [col for col in df_copy.columns if 'Wind Gust' in col][0]
    for col in [temp_col, humid_col, solar_col, rain_col, wind_col]:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
    
    initial_rows = len(df_copy)
    df_copy = df_copy.dropna(subset=[temp_col, humid_col, solar_col, rain_col, wind_col])
    
    daily = df_copy.groupby(['Field-Location', 'date']).agg({
        temp_col: ['mean', 'max', 'min'],
        humid_col: 'mean',
        solar_col: ['sum', 'max'],
        rain_col: 'sum',
        wind_col: 'max'
    }).round(2)
    
    colsList = []

    for col in daily.columns.values:
        processed_col_name = '_'.join(col).strip()
        colsList.append(processed_col_name)
    daily.columns = colsList
    
    daily = daily.rename(columns={
        f'{temp_col}_mean': 'Temperature.C_mean',
        f'{temp_col}_max': 'Temperature.C_max',
        f'{temp_col}_min': 'Temperature.C_min',
        f'{humid_col}_mean': 'Relative_Humidity_perc_mean',
        f'{solar_col}_sum': 'Solar_Radiation_sum',
        f'{solar_col}_max': 'Solar_Radiation_max',
        f'{rain_col}_sum': 'Rainfall_mm_sum',
        f'{wind_col}_max': 'Wind_Gust_m_s_max'
    })
    
    daily = daily.reset_index()
    daily['year'] = year
    
    dailyWeather_list.append(daily)
weather_daily = pd.concat(dailyWeather_list, ignore_index=True)


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

vcf_file = "inbreds_G2F_2014-2023_437k.vcf"

full_phenotypic_df[['female_parent', 'male_parent']] = full_phenotypic_df['Pedigree'].str.split('/', expand=True)

female_parents_list = full_phenotypic_df['female_parent'].unique().tolist()
callset = allel.read_vcf(vcf_file, samples=female_parents_list, fields=['calldata/GT', 'variants/CHROM', 'variants/POS'])

genotypes = allel.GenotypeArray(callset['calldata/GT'])
snp_matrix = genotypes.to_n_alt().T

imputer = SimpleImputer(strategy='mean')
snp_matrix_imputed = imputer.fit_transform(snp_matrix)

selector = VarianceThreshold(threshold=0.01) 
snp_matrix_filtered = selector.fit_transform(snp_matrix_imputed)

pca = PCA(n_components=50)
snp_pca = pca.fit_transform(snp_matrix_filtered)

num_clusters = 15
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(snp_pca)

loaded_samples = callset['samples']
cluster_map = dict(zip(loaded_samples, clusters))
full_phenotypic_df['female_cluster'] = full_phenotypic_df['female_parent'].map(cluster_map)

/Users/aaryamanbisht/Library/Python/3.11/lib/python/site-packages/allel/io/vcf_read.py:1621: UserWarning: some samples not found, will be ignored: '6046-BECKS', 'A639', 'AGRO NAUT', 'ARPA W22 (X17EA)', 'BAROS', 'D54VC14 RIB', 'D54VC14RIB', 'DKC29_89RIB', 'DKC48_95RIB', 'DKC49-72RIB', 'DKC57-23RIB', 'DKC62-08', 'DKC64-65 RIB', 'DKC64-65RIB', 'DKC67-44', 'Dyna-Gro D54VC14 RIB', 'GEMN-0096_PHK76_0014', 'KELITIAS', 'LH123HT', 'LOCAL_CHECK', 'LOCAL_CHECK_1', 'LOCAL_CHECK_2', 'LOCAL_CHECK_3', 'LOCAL_CHECK_4', 'LOCAL_CHECK_5', 'MO44_PHW65_0011', 'MO44_PHW65_0110', 'MO44_PHW65_0252', 'MO44_PHW65_0256', 'MO44_PHW65_0349', 'NC368', 'PHG80xPHP02', 'PHN11_PHW65_0032', 'PHN11_PHW65_0094', 'PHN11_PHW65_0109', 'PHN11_PHW65_0407', 'PHN11_PHW65_0505', 'PHW65_MOG_0060', 'PHW65_MOG_0067', 'PHW65_MOG_0068', 'PHW65_MOG_0075', 'PHW65_MOG_0115', 'PHW65_MOG_0119', 'PHW65_MOG_0124', 'PHW65_MOG_0140', 'PHW65_MOG_0157', 'PHW65_MOG_0192', 'PHW65_MOG_0195', 'PHW65_MOG_0200', 'PHW65_MOG_0206', 'PHW65_MOG_0221', 'PH

KeyError: 'samples'

In [ ]:
import pickle
import numpy as np
import pandas as pd


valid_years = ['2015', '2019', '2020', '2021', '2022', '2023']
valid_df = full_phenotypic_df[full_phenotypic_df['year'].isin(valid_years)]

model_data = []
missing_weather = 0

for idx, row in valid_df.iterrows():
    if idx > 0 and idx % 5000 == 0:
        print(f"Processed {idx}/{len(valid_df)} hybrids...")
    
    weather_seq = get_hybrid_weather
    (row, weather_daily)
    
    if weather_seq is None:
        missing_weather += 1
        continue
    
    model_data.append({
        'pedigree': row['Pedigree'],
        'male_parent': row['male_parent'],
        'female_cluster': int(row['female_cluster']) if pd.notna(row['female_cluster']) else -1,
        'plant_height': row['Plant Height [cm]'],
        'ear_height': row['Ear Height [cm]'],
        'anthesis': row['Anthesis [days]'],
        'silking': row['Silking [days]'],
        'stand_count': row['Stand Count [# of plants]'],
        'yield': row['Grain Yield (bu/A)'],
        'weather_sequence': weather_seq.astype(np.float32), 
        'location': row['Field-Location'],
        'year': row['year']
    })

coverage = (len(model_data) / (len(model_data) + missing_weather)) * 100

with open('model_ready_data.pkl', 'wb') as f:
    pickle.dump(model_data, f)

baseline_records = []
for d in model_data:
    seq = d['weather_sequence']
    
    record = {
        'pedigree': d['pedigree'],
        'male_parent': d['male_parent'],
        'female_cluster': d['female_cluster'],
        'plant_height': d['plant_height'],
        'ear_height': d['ear_height'],
        'anthesis': d['anthesis'],
        'silking': d['silking'],
        'stand_count': d['stand_count'],
        'yield': d['yield'],
        'location': d['location'],
        'year': d['year'],
        'season_length': len(seq),
        
        'temp_mean_mean': seq[:, 0].mean(),
        'temp_mean_max': seq[:, 0].max(),
        'temp_mean_min': seq[:, 0].min(),
        'temp_max_mean': seq[:, 1].mean(),
        'temp_max_max': seq[:, 1].max(),
        'temp_min_mean': seq[:, 2].mean(),
        'temp_min_min': seq[:, 2].min(),
        'humidity_mean': seq[:, 3].mean(),
        'humidity_max': seq[:, 3].max(),
        'solar_sum_mean': seq[:, 4].mean(),
        'solar_sum_max': seq[:, 4].max(),
        'solar_sum_total': seq[:, 4].sum(),
        'solar_max_mean': seq[:, 5].mean(),
        'solar_max_max': seq[:, 5].max(),
        'rain_sum_mean': seq[:, 6].mean(),
        'rain_sum_max': seq[:, 6].max(),
        'rain_sum_total': seq[:, 6].sum(),
        'wind_max_mean': seq[:, 7].mean(),
        'wind_max_max': seq[:, 7].max(),
    }
    baseline_records.append(record)

baseline_df = pd.DataFrame(baseline_records)
baseline_df.to_csv('baseline_data.csv', index=False)

print(f"Saved baseline_data.csv | Shape: {baseline_df.shape}")

Processing 39469 hybrid records...


NameError: name 'get_hybrid_weather' is not defined

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

baseline_df = pd.read_csv('baseline_data.csv')
baseline_df = baseline_df.dropna()
print(f"Loaded {len(baseline_df)} records after dropping NAs")

FileNotFoundError: [Errno 2] No such file or directory: 'baseline_data.csv'

In [ ]:
target = 'yield'
features = [col for col in baseline_df.columns if col not in ['pedigree', 'yield', 'location', 'year']]

X = baseline_df[features].copy()
y = baseline_df[target].copy()

le_male = LabelEncoder()
X['male_parent_encoded'] = le_male.fit_transform(X['male_parent'])
X['female_cluster'] = X['female_cluster'].astype(int)
X = X.drop(columns=['male_parent'])

print(f"Feature matrix: {X.shape} | Features: {X.columns.tolist()}")

In [ ]:
baseline_df = pd.read_csv('baseline_data_corrected.csv')

target = 'yield'
features = [col for col in baseline_df.columns if col not in ['pedigree', 'yield', 'location']]

X = baseline_df[features].copy()
y = baseline_df[target].copy()

le_male = LabelEncoder()
X['male_parent_encoded'] = le_male.fit_transform(X['male_parent'])
X['female_cluster'] = X['female_cluster'].astype(int)
X = X.drop(columns=['male_parent'])

train_years = [2019, 2020, 2021]
test_years = [2022, 2023]

train_idx = baseline_df['year'].isin(train_years)
test_idx = baseline_df['year'].isin(test_years)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(X_train)} samples {train_years} | Test: {len(X_test)} samples {test_years}")

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
# Baseline 1: Linear Regression
linear_reg = LinearRegression()
liner_reg.fit(X_train_scaled, y_train)
y_pred_lr = linear_reg.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression | RMSE: {rmse_lr:.2f}  MAE: {mae_lr:.2f}  R2: {r2_lr:.4f}")

coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': linear_reg.coef_
}).sort_values('coefficient', key=abs, ascending=False)
print(coef_df.head(10))

print(f"Yield | mean: {y.mean():.2f}  std: {y.std():.2f}  min: {y.min():.2f}  max: {y.max():.2f}")

correlations = pd.DataFrame({
    'feature': X.columns,
    'correlation': [X[col].corr(y) for col in X.columns]
}).sort_values('correlation', ascending=False)

print(correlations.head(10))

In [ ]:
# Baseline 2: Ridge Regression
alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]
ridge = RidgeCV(alphas=alphas, cv=5)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Ridge (alpha={ridge.alpha_:.1f}) | RMSE: {rmse_ridge:.2f}  MAE: {mae_ridge:.2f}  R2: {r2_ridge:.4f}")

ridge_coef = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': ridge.coef_
}).sort_values('coefficient', key=abs, ascending=False)
print(ridge_coef.head(10))

In [ ]:
# Bonus Baseline: Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest | RMSE: {rmse_rf:.2f}  MAE: {mae_rf:.2f}  R2: {r2_rf:.4f}")

rf_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(rf_importance.head(10))

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Random Forest'],
    'RMSE': [312.88, rmse_ridge, rmse_rf],
    'R2': [-23.75, r2_ridge, r2_rf]
})
print(results)